Group 1

Name: Duk Pagnarith, Leng Devid, Sophorn Rachana


# TP 2

### Exercise 1:  Build the CSVReader Class (OOP Foundation)


In [1]:
import pandas as pd

In [36]:
# Create Class

class CSVReader:
    def __init__ (self, file_path):
        self.file_path = file_path
        self.data = None
    
    def read(self):
        self.data = pd.read_csv(self.file_path)
        return self.data

    def preview(self, n=5):
        print(self.data.head(n))


In [37]:
reader = CSVReader("Data/sample_data (2).csv")

In [38]:
reader.read()

,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,NaN,172.0,NaN,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,NaN
3,4,David,NaN,NaN,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,NaN
6,7,Grace,NaN,162.0,60.0,Chicago,92.0


In [39]:
reader.preview(n=5)

   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
3   4    David   NaN        NaN       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0


## TP02 Exercise 2: Handling missing values with the Strategy pattern


## 1. Import the tools and locate the CSV

In [ ]:
from pathlib import Path
from abc import ABC, abstractmethod

import pandas as pd
from IPython.display import display

## 2. Inspect the original data

In [ ]:
INPUT_PATH = Path("./sample_data (2).csv")

df = pd.read_csv(INPUT_PATH)
original_df = df.copy(deep=True)

display(df)

,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,NaN,172.0,NaN,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,NaN
3,4,David,NaN,NaN,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,NaN
6,7,Grace,NaN,162.0,60.0,Chicago,92.0


In [ ]:
print("Shape:", df.shape)

Shape: (7, 7)


In [ ]:
display(df.isna().sum())

id           0
name         0
age          3
height_cm    1
weight_kg    1
city         0
score        2
dtype: int64

## 3. Define the common strategy interface

In [ ]:
class MissingValueStrategy(ABC):
    """Common interface for missing-value handling strategies."""

    @abstractmethod
    def handle(self, df):
        """Return a cleaned DataFrame without modifying the input."""
        pass

## 4. Implement DropMissing

In [ ]:
class DropMissing(MissingValueStrategy):
    def handle(self, df):
        return df.dropna().copy()

## 5. Implement FillMean

In [ ]:
class FillMean(MissingValueStrategy):
    def handle(self, df):
        result = df.copy()
        numeric_columns = result.select_dtypes(include="number").columns

        for column in numeric_columns:
            mean_value = result[column].mean()
            result[column] = result[column].fillna(mean_value)

        return result

## 6. Implement FillMode

In [ ]:
class FillMode(MissingValueStrategy):
    def handle(self, df):
        result = df.copy()

        for column in result.columns:
            modes = result[column].mode()
            if not modes.empty:
                result[column] = result[column].fillna(modes.iloc[0])

        return result

## 7. Create DataCleaner

In [ ]:
class DataCleaner:
    def __init__(self, strategy):
        self.strategy = strategy

    def clean(self, df):
        return self.strategy.handle(df)


cleaner = DataCleaner(DropMissing())
cleaned_df = cleaner.clean(df)
display(cleaned_df)

,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
4,5,Eva,27.0,160.0,55.0,New York,88.0


In [ ]:
class DataCleaner:
    def __init__(self, strategy):
        self.strategy = strategy

    def clean(self, df):
        return self.strategy.handle(df)


cleaner = DataCleaner(FillMean())
cleaned_df = cleaner.clean(df)
display(cleaned_df)

,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.00,165.0,68.0,New York,85.0
1,2,Bob,30.25,172.0,70.0,Los Angeles,90.0
2,3,Charlie,35.00,168.0,72.0,Chicago,86.0
3,4,David,30.25,167.0,80.0,Houston,75.0
4,5,Eva,27.00,160.0,55.0,New York,88.0
5,6,Frank,30.00,175.0,85.0,Los Angeles,86.0
6,7,Grace,30.25,162.0,60.0,Chicago,92.0


In [ ]:
class DataCleaner:
    def __init__(self, strategy):
        self.strategy = strategy

    def clean(self, df):
        return self.strategy.handle(df)


cleaner = DataCleaner(FillMode())
cleaned_df = cleaner.clean(df)
display(cleaned_df)

,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,27.0,172.0,55.0,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,75.0
3,4,David,27.0,160.0,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,75.0
6,7,Grace,27.0,162.0,60.0,Chicago,92.0


## 8. Switch the strategy on the same cleaner

In [ ]:
cleaner = DataCleaner(FillMean())
mean_result = cleaner.clean(df)

cleaner.strategy = DropMissing()
drop_result = cleaner.clean(df)

cleaner.strategy = FillMode()
mode_result = cleaner.clean(df)

display(pd.DataFrame({
    "name": df["name"],
    "original_age": df["age"],
    "mean_filled_age": mean_result["age"],
    "mode_filled_age": mode_result["age"],
}))

,name,original_age,mean_filled_age,mode_filled_age
0,Alice,29.0,29.00,29.0
1,Bob,NaN,30.25,27.0
2,Charlie,35.0,35.00,35.0
3,David,NaN,30.25,27.0
4,Eva,27.0,27.00,27.0
5,Frank,30.0,30.00,30.0
6,Grace,NaN,30.25,27.0


# Decorator Pattern: Logging and Timing

## Objective

Learn how to add reusable functionality to existing methods without changing their original implementation, and understand how this supports the **Open/Closed Principle**.

In this notebook, we use a random employee CSV dataset and clean it with pandas. The dataset is only a practical example; the main topic is decorators.

## 1. What is a decorator?

A **decorator** is a function that wraps another function or method to add behavior before or after it runs.

For example, a data-cleaning method can clean a DataFrame. Decorators can add logging and execution-time measurement around that method without putting those extra tasks into the cleaning logic.

- `@log_action`: logs when a method starts and finishes.
- `@log_time`: measures how long a method takes.
- `@wraps`: preserves the wrapped function's metadata, such as its name and docstring.

Conceptually:

**Original method → decorator wrapper → original behavior + extra functionality**

## 2. The Open/Closed Principle

The **Open/Closed Principle** says that software entities should be:

- **Open for extension:** new behavior can be added.
- **Closed for modification:** existing, stable code should not need to change to add that behavior.

Here, `DataCleaner.clean()` focuses on cleaning data. Logging and timing are added with decorators, so the cleaning logic does not need to be rewritten to include those features.

Decorators are one way to support this principle; they do not guarantee that every design is automatically closed to modification.

## 3. Create the decorators

### `@log_action`

This decorator prints a start message before calling the original function and a finish message afterward. The `finally` block ensures the finish message is printed even if the wrapped function raises an exception.

### `@log_time`

This decorator records a start time, runs the original function, then calculates elapsed time with `time.perf_counter()`.

### `@wraps(func)`

Without `@wraps(func)`, the decorated function can appear to have the name `wrapper` and may lose its original docstring. `@wraps` preserves useful metadata from the original function.

In [ ]:
import time
from functools import wraps

def log_action(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"[START] {func.__name__}")
        try:
            return func(*args, **kwargs)
        finally:
            print(f"[FINISH] {func.__name__}")
    return wrapper


def log_time(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"[TIME] {func.__name__} took {elapsed:.4f} seconds")
        return result
    return wrapper

## 4. Generate and load the random dataset

The CSV contains intentionally imperfect employee data, such as missing values, inconsistent capitalization, invalid ages or salaries, and duplicate rows. These issues give us something realistic to clean.

Place `random_employee_data.csv` in the same folder as this notebook before running the next cell.

In [ ]:
import pandas as pd

df = pd.read_csv("random_employee_data.csv")

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (32, 6)


,employee_id,name,age,department,salary_usd,city
0,1,BOB,19.0,HR,1768.0,Siem Reap
1,2,george,20.0,Finance,311.0,Siem Reap
2,3,george,32.0,finance,NaN,Battambang
3,4,Charlie,NaN,HR,NaN,Phnom Penh
4,5,Hannah,52.0,Sales,505.0,Phnom Penh


## 5. Build a data cleaner with decorators

The `clean()` method contains the data-cleaning rules. The decorators add logging and timing externally.

Decorator order matters: with `@log_action` above `@log_time`, the logging wrapper surrounds the timing wrapper. The start message appears first, the timing message appears after the method runs, and the finish message appears last.

In [ ]:
class DataCleaner:
    @log_action
    @log_time
    def clean(self, df):
        """Return a cleaned copy of the input DataFrame."""
        df = df.copy()

        print(f"Rows before cleaning: {len(df)}")

        # Remove duplicate rows
        df = df.drop_duplicates()

        # Standardize text columns
        for col in ["name", "department", "city"]:
            df[col] = df[col].astype("string").str.strip()

        df["name"] = df["name"].str.title()
        df["department"] = df["department"].str.title()
        df["city"] = df["city"].str.title()

        # Treat impossible values as missing
        df.loc[~df["age"].between(18, 100), "age"] = pd.NA
        df.loc[df["salary_usd"] < 0, "salary_usd"] = pd.NA
        df.loc[df["salary_usd"] > 100_000, "salary_usd"] = pd.NA

        # Fill missing text fields
        for col in ["name", "department", "city"]:
            df[col] = df[col].fillna("Unknown")

        # Fill missing numeric values with the median
        df["age"] = df["age"].fillna(df["age"].median())
        df["salary_usd"] = df["salary_usd"].fillna(
            df["salary_usd"].median()
        )

        # Keep output types practical
        df["age"] = df["age"].round().astype(int)
        df["salary_usd"] = df["salary_usd"].round(2)

        print(f"Rows after cleaning: {len(df)}")
        return df

## 6. Run the cleaning method

The original DataFrame is passed to `clean()`. The method creates a copy, so the raw DataFrame remains unchanged.

In [ ]:
cleaner = DataCleaner()
cleaned_df = cleaner.clean(df)

display(cleaned_df.head())

[START] clean
Rows before cleaning: 32
Rows after cleaning: 30
[TIME] clean took 0.0030 seconds
[FINISH] clean


,employee_id,name,age,department,salary_usd,city
0,1,Bob,19,Hr,1768.0,Siem Reap
1,2,George,20,Finance,311.0,Siem Reap
2,3,George,32,Finance,1196.0,Battambang
3,4,Charlie,42,Hr,1196.0,Phnom Penh
4,5,Hannah,52,Sales,505.0,Phnom Penh


## 7. Compare before and after cleaning

We can check the dataset dimensions and missing values to see the effect of the cleaning rules.

In [ ]:
print("Original shape:", df.shape)
print("Cleaned shape:", cleaned_df.shape)

print("\nMissing values before cleaning:")
display(df.isna().sum().to_frame("missing_before"))

print("\nMissing values after cleaning:")
display(cleaned_df.isna().sum().to_frame("missing_after"))

Original shape: (32, 6)
Cleaned shape: (30, 6)

Missing values before cleaning:


,missing_before
employee_id,0
name,3
age,1
department,5
salary_usd,3
city,8



Missing values after cleaning:


,missing_after
employee_id,0
name,0
age,0
department,0
salary_usd,0
city,0


## 8. Why use `@wraps`?

`@wraps(func)` helps preserve the original function's metadata after decoration. This is useful for debugging, documentation, and inspection.

For example, we can inspect the decorated method's name and docstring.

In [ ]:
print("Method name:", cleaner.clean.__name__)
print("Method docstring:", cleaner.clean.__doc__)

Method name: clean
Method docstring: Return a cleaned copy of the input DataFrame.


## 9. Key takeaways

1. **Decorators wrap functions or methods** to add behavior.
2. **`@log_action`** adds start and finish messages.
3. **`@log_time`** measures elapsed execution time using `time.perf_counter()`.
4. **`@wraps(func)`** preserves metadata such as the original name and docstring.
5. **Reusability:** the same decorators can be applied to methods in different classes.
6. **Open/Closed Principle:** we can extend methods with logging and timing without inserting that code into the original data-cleaning logic.

The CSV cleaning is the example. The main lesson is how decorators add reusable functionality around existing code.

In [ ]:
import time
import functools
from abc import ABC, abstractmethod

import pandas as pd

In [ ]:
# from excercise 3
def log_action(func):
    @functools.wraps(func)
    def wrapper(self, *args, **kwargs):
        print(f"[LOG] Starting {self.__class__.__name__}.{func.__name__}")
        result = func(self, *args, **kwargs)
        print(f"[LOG] Finished {self.__class__.__name__}.{func.__name__}")
        return result
    return wrapper
 
 
def log_time(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        print(f"[TIME] {func.__name__} took {time.perf_counter() - start:.4f}s")
        return result
    return wrapper

In [ ]:
# create an abstract class
class DataTransform(ABC):
    @abstractmethod
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        pass

In [ ]:
# subclass: NormalizeColumns
class NormalizeColumns(DataTransform):
    """Min-max scales numeric columns to the range [0, 1]."""

    def __init__(self, columns=None):
        self.columns = columns

    @log_action
    @log_time
    def apply(self, df):
        df = df.copy()
        cols = self.columns or df.select_dtypes(include="number").columns.tolist()
        for col in cols:
            col_min, col_max = df[col].min(), df[col].max()
            if col_max == col_min:
                df[col] = 0.0
            else:
                df[col] = (df[col] - col_min) / (col_max - col_min)
        return df

# subclass: RemoveDuplicates
class RemoveDuplicates(DataTransform):
    """Drops duplicate rows (optionally based on a subset of columns)."""
 
    def __init__(self, subset=None):
        self.subset = subset
 
    @log_action
    @log_time
    def apply(self, df):
        before = len(df)
        df = df.drop_duplicates(subset=self.subset).reset_index(drop=True)
        print(f"   -> removed {before - len(df)} duplicate row(s)")
        return df

# subclass: StandardizeText
class StandardizeText(DataTransform):
    """Strips spaces and unifies the case of text columns."""
 
    def __init__(self, columns=None, case="title"):
        self.columns = columns  # None -> all text columns
        self.case = case        # "lower", "upper" or "title"
 
    @log_action
    @log_time
    def apply(self, df):
        df = df.copy()
        cols = self.columns or df.select_dtypes(include="object").columns.tolist()
        for col in cols:
            text = df[col].astype("string").str.strip().str.replace(r"\s+", " ", regex=True)
            df[col] = getattr(text.str, self.case)()
        return df

In [ ]:
# create Factory
class TransformFactory:
    """Returns a transformation object from its name, so client code
    never has to instantiate the concrete classes itself."""
 
    _registry = {
        "normalize": NormalizeColumns,
        "remove_duplicates": RemoveDuplicates,
        "standardize_text": StandardizeText,
    }
 
    @classmethod
    def create(cls, name: str, **kwargs) -> DataTransform:
        key = name.strip().lower()
        if key not in cls._registry:
            raise ValueError(
                f"Unknown transformation '{name}'. Available: {list(cls._registry)}"
            )
        return cls._registry[key](**kwargs)
 
    @classmethod
    def register(cls, name: str, transform_cls):
        """Add a new transformation without modifying the factory (Open/Closed)."""
        if not issubclass(transform_cls, DataTransform):
            raise TypeError("transform_cls must inherit from DataTransform")
        cls._registry[name.lower()] = transform_cls

In [ ]:
# ---------------------------------------------------------------
# test
# ---------------------------------------------------------------
if __name__ == "__main__":
    df = pd.read_csv("sample_data (2).csv")
 
    # Add messy text + a duplicate row so every transformation has something to do
    df.loc[len(df)] = df.loc[0]
    df.loc[1, "city"] = "  los   angeles "
    print("Original data:\n", df, "\n")
 
    steps = [
        ("remove_duplicates", {}),
        ("standardize_text", {"columns": ["name", "city"]}),
        ("normalize", {"columns": ["age", "height_cm", "weight_kg", "score"]}),
    ]
 
    for name, params in steps:
        transform = TransformFactory.create(name, **params)  # the factory decides
        df = transform.apply(df)                             # polymorphism
        print()
 
    print("Transformed data:\n", df.round(3))

Original data:
    id     name   age  height_cm  weight_kg              city  score
0   1    Alice  29.0      165.0       68.0          New York   85.0
1   2      Bob   NaN      172.0        NaN    los   angeles    90.0
2   3  Charlie  35.0      168.0       72.0           Chicago    NaN
3   4    David   NaN        NaN       80.0           Houston   75.0
4   5      Eva  27.0      160.0       55.0          New York   88.0
5   6    Frank  30.0      175.0       85.0       Los Angeles    NaN
6   7    Grace   NaN      162.0       60.0           Chicago   92.0
7   1    Alice  29.0      165.0       68.0          New York   85.0 

[LOG] Starting RemoveDuplicates.apply
   -> removed 1 duplicate row(s)
[TIME] apply took 0.0012s
[LOG] Finished RemoveDuplicates.apply

[LOG] Starting StandardizeText.apply
[TIME] apply took 0.0020s
[LOG] Finished StandardizeText.apply

[LOG] Starting NormalizeColumns.apply
[TIME] apply took 0.0021s
[LOG] Finished NormalizeColumns.apply

Transformed data:
    id     n

---
## Exercise 5: Build a Full Cleaning Pipeline (Template Method Pattern)

### Objectives & Concepts
- **Template Method Pattern:** The base class defines the skeleton of the algorithm (`run()`), and subclasses implement the individual steps.
- **Composition:** The pipeline reuses `CSVReader`, `DataCleaner` and the `DataTransform` objects from the previous exercises.
- **Abstraction:** `@abstractmethod` forces every pipeline to implement `load`, `clean`, `transform` and `save`.

### Instructions:
1. Create an abstract class `DataPipeline` with a `run()` method that calls: `load()`, `clean()`, `transform()`, `save()`.
2. Create `CSVDataPipeline` that implements each step using the previous classes.

In [ ]:
class DataPipeline(ABC):
    def run(self):                      # fixed order of steps
        df = self.load()
        df = self.clean(df)
        df = self.transform(df)
        self.save(df)
        return df

    @abstractmethod
    def load(self): pass
    @abstractmethod
    def clean(self, df): pass
    @abstractmethod
    def transform(self, df): pass
    @abstractmethod
    def save(self, df): pass


class CSVDataPipeline(DataPipeline):
    def __init__(self, input_path, output_path, cleaner, transforms):
        self.reader = CSVReader(input_path)
        self.output_path = output_path
        self.cleaner = cleaner
        self.transforms = transforms

    def load(self):
        return self.reader.read()

    def clean(self, df):
        return self.cleaner.clean(df)

    def transform(self, df):
        for t in self.transforms:
            df = t.apply(df)
        return df

    def save(self, df):
        df.to_csv(self.output_path, index=False)


pipeline = CSVDataPipeline(
    "sample_data.csv",
    "cleaned_sample_data.csv",
    DataCleaner(FillMean()),
    [TransformFactory.get_transform("remove_duplicates"),
     TransformFactory.get_transform("standardize_text"),
     TransformFactory.get_transform("normalize")],
)
result = pipeline.run()
print(result.round(2))

NameError: name 'DataCleaner' is not defined